# Radar through the winter — Baltic sea ice with Sentinel-1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/09_sar_baltic_ice.ipynb)

Every animation so far used **optical** sensors, and all of them share one
weakness: clouds. A Baltic winter is their worst case — weeks of overcast and
eight hours of grey daylight. **Synthetic-aperture radar (SAR)** does not care:
it brings its own illumination and its microwaves pass through clouds, so
Sentinel-1 images the sea ice in the Bodden lagoons on every single pass, day
and night, December through April.

This notebook animates the **winter 2025/26** over the Rügen–Usedom coast.
Radar is a package product like any other (`sensor="sentinel1"`,
`index="sar_vv"` or `"sar_vh"`), so the animation is one `animate()` call with
the pooling, the GeoTIFF export and the frame furniture the optical products
get. What stays explicit is the look into the archive: which modes and
polarisations actually cover this coast is a question for the data, not for a
tutorial's assumption.

## 1 · Install (Colab only)

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code

## 2 · Authenticate

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · The scene and the season

The AOI spans the shallow lagoons between Rügen and Usedom — the Greifswalder
Bodden freezes in most cold spells while the open Baltic beside it stays liquid,
which is exactly the contrast radar shows well.

In [ ]:
from pathlib import Path
import ee
from gee_animation import aoi

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "aoi").is_dir()), None)
if REPO is not None:
    REGION = str(REPO / "docs" / "aoi" / "balticsea_ice" / "aoi.geojson")
else:   # Colab: fetch the same file
    import urllib.request
    RAW = (f"https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/{BRANCH}"
           "/docs/aoi/balticsea_ice/aoi.geojson")
    urllib.request.urlretrieve(RAW, "aoi.geojson")
    REGION = "aoi.geojson"

START, END = "2025-12-01", "2026-05-01"      # the ice season, end exclusive

region_geom = aoi.parse({"geojson": REGION})
bbox = region_geom.bounds().coordinates().get(0).getInfo()
print("AOI corners:", [[round(c, 2) for c in p] for p in bbox[:-1]])

## 4 · Ask the archive before assuming anything

Sentinel-1 scenes differ by **instrument mode** (IW over most land and coasts,
EW in the Arctic) and **polarisation** (VV, VH, HH …). Which combination
actually covers *this* AOI *this* winter is a question for the archive, not for
a tutorial's assumption:

In [ ]:
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(region_geom)
      .filterDate(START, END))

combos = s1.aggregate_histogram(
    "instrumentMode").getInfo()
pols = s1.aggregate_histogram("transmitterReceiverPolarisation").getInfo()
print("scenes by mode:        ", combos)
print("scenes by polarisation:", pols)

BAND = "VV"          # chosen from the inventory above: IW VV+VH dominates here
s1 = (s1.filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", BAND))
        .select(BAND))
print(f"usable {BAND} scenes:", s1.size().getInfo())

## 5 · Ten-day backscatter mosaics, one call

Sea ice moves and grows on a scale of days, so monthly frames would blur the
story; `cadence="10day"` keeps the motion. Each frame is the **median of all
passes in its ten days** — 12 to 21 here — which is also the speckle filter:
radar speckle is multiplicative noise that a median over many looks averages
out. `S1_GRD` delivers backscatter in **dB** (a log scale); the median of dB
values is fine for a picture, just not the mean of the physical power.

`EXPORT_GEOTIFF = True` additionally writes one float32 GeoTIFF per period
(the values, not the picture) under `out/geotiffs/baltic_ice_sar_vv/` — the
manual export switch for QGIS or your own threshold.

In [ ]:
from gee_animation.api import animate

EXPORT_GEOTIFF = False            # True: one float32 GeoTIFF (dB) per period, for GIS
radar = animate(REGION, sensor="sentinel1", index="sar_vv", start=START, end=END,
                cadence="10day", buffer_m=0, preset="720p", fps=2,
                name="baltic_ice_sar_vv", out_dir="out", project=PROJECT,
                chart=False, geotiffs=EXPORT_GEOTIFF)
radar                             # plays inline: 15 ten-day frames, December to April

## 6 · Reading the frames

Wind-roughened **open water scatters strongly** (bright); **smooth new ice is
dark**; ridged or snow-covered ice sits in between, and land is bright
year-round. Watch the Greifswalder Bodden and the Stettiner Haff: they go dark
in the February cold spell while the open Baltic beside them stays bright, and
they brighten again as the ice breaks up in March. A straight brightness step
across open water is not ice: it is the seam between an ascending and a
descending pass, whose incidence angles differ.

## 7 · What optical would have given you

The same AOI, the same winter, Sentinel-2. Counting granules under the series'
usual 70 % cloud filter gives flattering numbers: the AOI touches five 100 km
tiles seen from three orbits, so one clear day yields five or more granules, and a
granule that only clips the AOI's edge counts the same as a full view. So count
**days**, and only days whose granules together cover most of the AOI. Footprints
and the cloud percentage are metadata — one request fetches them all, the
geometry is done locally — and the case for radar makes itself:

Then the same animation again from optical through the same `animate()` call:
a cloud-masked true-colour median per 10-day period, next to the radar frame of
that period. White is cloud (or snow and ice) the mask let through, grey is
where ten days held no usable pixel at all.

In [ ]:
import json
from collections import defaultdict
from shapely.geometry import shape
from shapely.ops import unary_union, transform
from pyproj import Transformer

MIN_COVER = 0.7                                     # a pass must see this share of the AOI
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(region_geom).filterDate(START, END))
granules = s2.map(lambda im: ee.Feature(im.geometry(), {
    "date": im.date().format("YYYY-MM-dd"),
    "cloud": im.get("CLOUDY_PIXEL_PERCENTAGE")})).getInfo()["features"]   # one call, metadata only

to_m = Transformer.from_crs("EPSG:4326", "EPSG:32633", always_xy=True).transform   # UTM 33N, metres
aoi_m = transform(to_m, shape(json.load(open(REGION))["features"][0]["geometry"]))
by_day = defaultdict(list)
for f in granules:
    part = transform(to_m, shape(f["geometry"])).intersection(aoi_m)
    if not part.is_empty:
        by_day[f["properties"]["date"]].append((part, f["properties"]["cloud"]))

days = []                                           # (date, AOI share covered, area-weighted granule cloud %)
for d, parts in sorted(by_day.items()):
    cover = unary_union([p for p, _ in parts]).area / aoi_m.area
    cloud = sum(p.area * c for p, c in parts) / sum(p.area for p, _ in parts)
    if cover >= MIN_COVER:
        days.append((d, cover, cloud))
print(f"{len(by_day)} days with any granule, {len(days)} of them cover >= {MIN_COVER:.0%} of the AOI")
for m in sorted({d[:7] for d in by_day}):
    full = [x for x in days if x[0].startswith(m)]
    print(f"{m}: {len(full):2d} full passes, {sum(c <= 30 for _, _, c in full)} with <= 30 % cloud")
print("radar passes in the same window:", s1.size().getInfo())

In [ ]:
# The same ten-day frames from optical, through the same call: a cloud-masked
# true-colour median per period. Where no usable pixel exists in ten days the frame
# stays grey; what the mask lets through stays white — the honest picture of a Baltic winter.
optical = animate(REGION, sensor="sentinel2", index="rgb", start=START, end=END,
                  cadence="10day", buffer_m=0, preset="720p", fps=2,
                  name="baltic_ice_rgb", out_dir="out", project=PROJECT,
                  max_cloud_percent=100, region_max_cloud_percent=100, chart=False)

import re
import numpy as np
import imageio.v2 as imageio
from PIL import Image as PILImage
from IPython.display import Image as IPyImage, display

def by_label(anim):
    return {re.search(r"_(\d{4}-\d{2}(?:-\d{2})?)\.png$", p).group(1): p for p in anim.frames}

left, right = by_label(radar), by_label(optical)
pairs = []
for label, path in sorted(left.items()):          # radar left, optical right, same frame
    a = PILImage.open(path).convert("RGB")
    b = (PILImage.open(right[label]).convert("RGB").resize(a.size) if label in right
         else PILImage.new("RGB", a.size, (40, 40, 40)))
    sheet = PILImage.new("RGB", (a.width * 2, a.height))
    sheet.paste(a, (0, 0)); sheet.paste(b, (a.width, 0))
    pairs.append(np.array(sheet))
gif = Path("out") / "baltic_ice_radar_vs_optical.gif"
imageio.mimwrite(gif, pairs, duration=500, loop=0)
print(f"{len(left)} radar periods, {len(right)} with any optical pixels; wrote {gif}")
display(IPyImage(filename=str(gif)))

## Make it yours

- **VH instead of VV** (`index="sar_vh"`): cross-polarisation darkens open water
  further and makes deformed ice stand out.
- **A real pack-ice winter**: move the AOI to the Bothnian Bay (around
  `[20.5, 64.5, 25.5, 65.9]`) where the Baltic freezes every year, and re-run —
  the archive inventory cell will tell you if EW mode takes over (the product
  uses IW only).
- **Another season**: the 2024/25 or 2026/27 winter is one date change away.
- **Smoother motion**: the CLI config `config/sar_vv.example.yaml` renders the
  same season with ten generated frames between observations.
- Compare your animation to the
  [FMI ice charts](https://en.ilmatieteenlaitos.fi/ice-conditions) for the same
  weeks — the professional ground truth for this sea.
- The product itself is forty lines in `gee_animation/products.py`
  (`sentinel1`, `sar_vv`, `sar_vh`); a VH/VV ratio index would follow the
  `adding-a-product` checklist in `.claude/skills/`.